# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shenone77/flyrank-ml-internship/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

print(f"Pages with a positive hand-rule score: {df['hand_rule_score'].gt(0).sum():,}")

Pages with a positive hand-rule score: 17


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.800
Precision@50:  hand rule 0.680   vs   tree 0.740


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

### Client holdout and tied scores

Train the same depth-2 tree on 75% of clients and score both methods on the remaining clients. The hand rule uses its original thresholds. Precision@K uses input order for tied scores. Tie-averaged precision estimates what the result would be if the remaining review slots were assigned uniformly among pages with the cutoff score.

In [7]:
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit

clients = df["client_id"]
split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(split.split(X, y, groups=clients))
train_clients = set(clients.iloc[train_idx])
test_clients = set(clients.iloc[test_idx])
assert train_clients.isdisjoint(test_clients)
assert {"client_id", "content_id", "trend_direction", "trend_pct"}.isdisjoint(X.columns)

holdout_tree = clone(tree).fit(X.iloc[train_idx], y[train_idx])
test_labels = y[test_idx]
print(f"Train: {len(train_idx):,} pages from {len(train_clients)} clients.")
print(f"Test: {len(test_idx):,} pages from {len(test_clients)} clients.")
print(f"Declining share among held-out pages: {test_labels.mean():.1%}.")


def cutoff_metrics(scores, labels, k):
    scores = np.asarray(scores)
    order = np.argsort(-scores, kind="stable")
    cutoff = scores[order[k - 1]]
    above = scores > cutoff
    tied = scores == cutoff
    slots = k - above.sum()
    expected_hits = labels[above].sum() + slots * labels[tied].mean()
    return labels[order[:k]].mean(), expected_hits / k, tied.sum(), slots


assert cutoff_metrics([3, 2, 2, 1], np.array([1, 0, 1, 0]), 2) == (0.5, 0.75, 2, 1)
test_scores = {
    "Hand rule": df["hand_rule_score"].iloc[test_idx],
    "Depth-2 tree": holdout_tree.predict_proba(X.iloc[test_idx])[:, 1],
}

results = []
for k in (20, 50):
    for method, scores in test_scores.items():
        precision, tie_average, tied, slots = cutoff_metrics(scores, test_labels, k)
        results.append({
            "K": k,
            "Method": method,
            "Precision@K": precision,
            "Tie-averaged precision": tie_average,
            "Pages tied at cutoff": tied,
            "Slots filled from tie": slots,
        })

display(pd.DataFrame(results).round(3))

Train: 22,885 pages from 24 clients.
Test: 7,115 pages from 8 clients.
Declining share among held-out pages: 51.7%.


,K,Method,Precision@K,Tie-averaged precision,Pages tied at cutoff,Slots filled from tie
0,20,Hand rule,0.80,0.589,7112,17
1,20,Depth-2 tree,0.65,0.576,3764,20
2,50,Hand rule,0.72,0.545,7112,47
3,50,Depth-2 tree,0.62,0.576,3764,50


**Result in the saved run.** With input-order tie breaking, the hand rule reached 0.72 Precision@50 and the tree reached 0.62. Only three held-out pages had a positive hand-rule score, leaving 47 of its 50 review slots to fill from a zero-score tie. Under uniform tie breaking, expected Precision@50 was 0.545 for the rule and 0.576 for the tree. The small advantage depends on the review budget and tie policy. This split tests transfer to unseen clients on the starter snapshot. A future-decline forecast would need features measured before the outcome window.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.